In [1]:
# Input: A directed graph with weight matrix `weight' and a start vertex `s'.
# Output: An array `D' of distances as explained above.

graph = {
    'A': {'B': 4, 'C': 4},
    'B': {'A': 4, 'C': 2},
    'C': {'A': 4, 'B': 2, 'D': 3, 'E': 1, 'F': 6},
    'D': {'C': 3, 'F': 2},
    'E': {'C': 1, 'F': 3},
    'F': {'C': 6, 'D': 2, 'E': 3},
}

In [ ]:
# Graph to mermaid

mermaid = "flowchart LR\n"

for v, adj_list in graph.items():
    # Stating Node
    mermaid += " " * 4
    mermaid += f"{v}(({v}))\n"

    # adding Edges
    for u, w in adj_list.items():
        mermaid += " " * 4
        mermaid += f"{v}-- {w} -->{u}\n"

    mermaid += "\n"

print(mermaid)

```mermaid
flowchart LR
    A((A))
    A-- 4 -->B
    A-- 4 -->C

    B((B))
    B-- 4 -->A
    B-- 2 -->C

    C((C))
    C-- 4 -->A
    C-- 2 -->B
    C-- 3 -->D
    C-- 1 -->E
    C-- 6 -->F

    D((D))
    D-- 3 -->C
    D-- 2 -->F

    E((E))
    E-- 1 -->C
    E-- 3 -->F

    F((F))
    F-- 6 -->C
    F-- 2 -->D
    F-- 3 -->E
```

### Dijkstra's algorithm, Version 1.

In [37]:
# Dijkstra's algorithm v1.0

from typing import Dict
from prettytable import PrettyTable

def dijkstra(graph: Dict[str, Dict[str, int]], from_node: str, to_node: str):
    """_summary_

    Args:
        graph (Dict[str, Dict[str, int]]): _description_
    """
    print(f"Computing shortest paths from {from_node}\n")
    # Initializing values
    nodes = list(graph.keys())
    distance = {k: float("inf") for k in nodes}
    distance[from_node] = 0
    visited = {k: False for k in nodes}
    previous = {k: None for k in nodes}

    stage_i = 0
    stages = PrettyTable()
    stages.field_names = ["Stage"] + nodes
    while not all(visited.values()):
        # finding unvisited with lower estimate
        current_node = None
        lower_est = float('inf')
        for v in nodes:
            if distance[v] < lower_est and not visited[v]:
                current_node = v
                lower_est = distance[v]


        # Current state
        table = PrettyTable()
        table.field_names = [""] + nodes
        table.add_row(['Dist'] + list(distance.values()))
        table.add_row(['Vis.'] + list(visited.values()))
        table.add_row(['Prev'] + list(previous.values()))
        print(table)

        # Adding to stages
        stage_i += 1
        stages.add_row(
            [stage_i] + \
            [f"{distance[x]:>3} {'+' if visited[x] else ' '} {previous[x] if previous[x] else ' '}" for x in nodes]
        )

        visited[current_node] = True
        print(f"\nVertex {current_node} is now visited\n")

        dist = distance[current_node]
        for u, w in graph[current_node].items():
            new_dist = dist + w
            prev_dist = distance[u]
            if new_dist < prev_dist:
                print(f"Neighbour {u} has estimate decreased from {prev_dist} to {new_dist} taking a shortcut via {current_node}.")
                distance[u] = new_dist
                previous[u] = current_node
            else:
                print(f"Neighbour {u} has estimate unchanged.")

    # Current state
    table = PrettyTable()
    table.field_names = [""] + nodes
    table.add_row(['Dist'] + list(distance.values()))
    table.add_row(['Vis.'] + list(visited.values()))
    table.add_row(['Prev'] + list(previous.values()))
    print("\n", table, sep="")

    # Adding to stages
    stage_i += 1
    stages.add_row(
        [stage_i] + \
        [f"{distance[x]:>3} {'+' if visited[x] else ' '} {previous[x] if previous[x] else ' '}" for x in nodes]
    )

    print("\nEnd of Dijkstra's computation.")
    print("\n", stages, "\n", sep="")

    # Getting the shortest path
    short_path = [to_node]
    current_node = to_node
    while current_node != from_node:
        current_node = previous[current_node]
        short_path.insert(0, current_node)

    print(f"A shortest path from {from_node} to {to_node} is: {' '.join(short_path)}.")


dijkstra(graph, "A", "E")

Computing shortest paths from A

+------+-------+-------+-------+-------+-------+-------+
|      |   A   |   B   |   C   |   D   |   E   |   F   |
+------+-------+-------+-------+-------+-------+-------+
| Dist |   0   |  inf  |  inf  |  inf  |  inf  |  inf  |
| Vis. | False | False | False | False | False | False |
| Prev |  None |  None |  None |  None |  None |  None |
+------+-------+-------+-------+-------+-------+-------+

Vertex A is now visited

Neighbour B has estimate decreased from inf to 4 taking a shortcut via A.
Neighbour C has estimate decreased from inf to 4 taking a shortcut via A.
+------+------+-------+-------+-------+-------+-------+
|      |  A   |   B   |   C   |   D   |   E   |   F   |
+------+------+-------+-------+-------+-------+-------+
| Dist |  0   |   4   |   4   |  inf  |  inf  |  inf  |
| Vis. | True | False | False | False | False | False |
| Prev | None |   A   |   A   |  None |  None |  None |
+------+------+-------+-------+-------+-------+-------+

V

### Dijkstra's algorithm, Version 2. 
The time complexity of Dijkstra's algorithm can be
improved by making use of a priority queue (e.g., some form of heap) to keep track of which
node's distance estimate becomes tight next.

In [43]:
# Min-Priority Queue

class MinHeap:
    def __init__(self):
        self.heap = []

    # return the number of nodes in a heap
    def size(self):
            return len(self.heap)

    # check if the heap is empty
    def is_empty(self):
        return len(self.heap) == 0

    # string representation of a heap
    def __str__(self):
        return str(self.heap)

    # swap the heap values
    def swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    # calculate the index of a node's parent
    def parent(self, index):
        return (index - 1) // 2

    # calculate the index of a node's left child
    def left_child(self, index):
        return 2 * index + 1

    # calculate the index of a node's right_child
    def right_child(self, index):
        return 2 * index + 2

    # check if a node at a given index has a parent
    def has_parent(self, index):
        return self.parent(index) >= 0

    # check if a node at a given index has a left child
    def has_left_child(self, index):
        return self.left_child(index) < len(self.heap)

    # check if a node at a given index has a right child
    def has_right_child(self, index):
        return self.right_child(index) < len(self.heap)
       
    # heapify 
    def heapify(self, array):
        self.heap = array
        for i in range(len(self.heap) - 1, -1, -1):
            if  self.has_left_child(i):
                self.heapify_down(i)
        
    # heapify down
    def heapify_down(self, index):
        if not self.has_left_child(index):
            return
        smaller_child_index = self.left_child(index)
        if self.has_right_child(index): 
            if self.heap[self.right_child(index)] < self.heap[smaller_child_index]:
                smaller_child_index = self.right_child(index)
        if self.heap[index] > self.heap[smaller_child_index]:
            self.swap(index, smaller_child_index)
        if self.has_left_child(smaller_child_index):
            self.heapify_down(smaller_child_index)
    
    # heapify up
    def heapify_up(self, index):
        if self.has_parent(index): 
            parent_index = self.parent(index)
            # compare the value to its parent and swap if necessary
            if self.heap[index] < self.heap[self.parent(index)]:
                parent_index = self.parent(index)
                self.swap(index, parent_index)
                # run the code for parent                 
                if self.has_parent(parent_index):
                    self.heapify_up(parent_index)

    # insert into heap
    def insert(self, value):
        self.heap.append(value)
        index = len(self.heap) - 1
        while self.has_parent(index) and self.heap[index] < self.heap[self.parent(index)]:
            parent_index = self.parent(index)
            self.swap(index, parent_index)
            index = parent_index
    
    # extract min from heap
    def extract_min(self):
        min_value = self.heap[0]
        if self.size() <= 1:
            self.heap = []
            return min_value
        self.heap = [self.heap[-1]] + self.heap[1:-1]
        index = 0
        while True:
            if not self.has_left_child(index):
                break
            smaller_child_index = self.left_child(index)
            if self.has_right_child(index): 
                if self.heap[self.right_child(index)] < self.heap[self.left_child(index)]:
                    smaller_child_index = self.right_child(index)
            if self.heap[index] > self.heap[smaller_child_index]:
                self.swap(index, smaller_child_index)
            else:
                break 
            index = smaller_child_index
        return min_value
    
    # peek
    def peek(self):
        return self.heap[0]
    

class PriorityQueue(MinHeap):
    def __init__(self):
        super().__init__()

    def insert(self, priority, value):
        return super().insert((priority, value))
    
    # change the priority (key) of an element already in the priority queue
    def change_key(self, value, new_priority):
        # search for value
        for i in range(self.size()):
            # if value is found
            if self.heap[i][1] == value:
                # change priority
                old_priority = self.heap[i][0]
                self.heap[i] = (new_priority, value)
                # heapify after changing priority to maintain heap property.
                # we can simply call heapify but heapify_up/down is more efficient
                if new_priority < old_priority:
                    self.heapify_up(i)
                else:
                    self.heapify_down(i)
                return


In [47]:
# Priority Queue test

# create a priority queue
priority_queue = PriorityQueue()

# tasks with priorities
tasks = [(12, 'Task1'), (15, 'Task2'), (25, 'Task3'), (16, 'Task4'), (5, 'Task5')]
# insert into priority queue
print('Insert tasks into priority queue')
for priority, task in tasks:
    priority_queue.insert(task, priority)
print(priority_queue)
print('--------------')


# change the key of Task1 from 12 to 20
print('Change priority of task1 to 20')
priority_queue.change_key('Task1', 20)
print(priority_queue)
print('--------------')

# change the key of Task1 from 20 to 2
print('Change priority of task1 to 2')
priority_queue.change_key('Task1', 2)
print(priority_queue)
print('--------------')

# peek at the new element with the highest priority(min value)
print('Top priority task')
print(priority_queue.peek()) 
print('-----------')

# top priority task is complete
priority_queue.extract_min()
print('After removing top priority task')
print(priority_queue)


Insert tasks into priority queue
[('Task1', 12), ('Task2', 15), ('Task3', 25), ('Task4', 16), ('Task5', 5)]
--------------
Change priority of task1 to 20
[('Task1', 12), ('Task2', 15), ('Task3', 25), ('Task4', 16), ('Task5', 5)]
--------------
Change priority of task1 to 2
[('Task1', 12), ('Task2', 15), ('Task3', 25), ('Task4', 16), ('Task5', 5)]
--------------
Top priority task
('Task1', 12)
-----------
After removing top priority task
[('Task2', 15), ('Task4', 16), ('Task3', 25), ('Task5', 5)]


In [50]:
# Dijkstra's algorithm v2.0

from typing import Dict
from prettytable import PrettyTable

def dijkstra(graph: Dict[str, Dict[str, int]], from_node: str, to_node: str):
    """_summary_

    Args:
        graph (Dict[str, Dict[str, int]]): _description_
    """
    print(f"Computing shortest paths from {from_node}\n")
    # Initializing values
    nodes = list(graph.keys())
    distance = {k: float("inf") for k in nodes}
    visited = {k: False for k in nodes}
    previous = {k: None for k in nodes}

    priority_queue = PriorityQueue()
    distance[from_node] = 0
    priority_queue.insert(0, from_node)

    stage_i = 0
    stages = PrettyTable()
    stages.field_names = ["Stage"] + nodes

    while not priority_queue.is_empty():
        print(f"\nPriority Queue: {priority_queue}\n")
        # Lowest node
        current_node = priority_queue.extract_min()[1]

        # Current state
        table = PrettyTable()
        table.field_names = [""] + nodes
        table.add_row(['Dist'] + list(distance.values()))
        table.add_row(['Vis.'] + list(visited.values()))
        table.add_row(['Prev'] + list(previous.values()))
        print(table)

        # Adding to stages
        stage_i += 1
        stages.add_row(
            [stage_i] + \
            [f"{distance[x]:>3} {'+' if visited[x] else ' '} {previous[x] if previous[x] else ' '}" for x in nodes]
        )

        visited[current_node] = True
        print(f"\nVertex {current_node} is now visited\n")

        dist = distance[current_node]
        for u, w in graph[current_node].items():
            new_dist = dist + w
            prev_dist = distance[u]
            if new_dist < prev_dist:
                print(f"Neighbour {u} has estimate decreased from {prev_dist} to {new_dist} taking a shortcut via {current_node}.")
                distance[u] = new_dist
                previous[u] = current_node

                # add to priority queue
                priority_queue.insert(new_dist, u)

            else:
                print(f"Neighbour {u} has estimate unchanged.")

    # Current state
    table = PrettyTable()
    table.field_names = [""] + nodes
    table.add_row(['Dist'] + list(distance.values()))
    table.add_row(['Vis.'] + list(visited.values()))
    table.add_row(['Prev'] + list(previous.values()))
    print("\n", table, sep="")

    # Adding to stages
    stage_i += 1
    stages.add_row(
        [stage_i] + \
        [f"{distance[x]:>3} {'+' if visited[x] else ' '} {previous[x] if previous[x] else ' '}" for x in nodes]
    )

    print("\nEnd of Dijkstra's computation.")
    print("\n", stages, "\n", sep="")

    # Getting the shortest path
    short_path = [to_node]
    current_node = to_node
    while current_node != from_node:
        current_node = previous[current_node]
        short_path.insert(0, current_node)

    print(f"A shortest path from {from_node} to {to_node} is: {' '.join(short_path)}.")


dijkstra(graph, "A", "E")

Computing shortest paths from A


Priority Queue: [(0, 'A')]

+------+-------+-------+-------+-------+-------+-------+
|      |   A   |   B   |   C   |   D   |   E   |   F   |
+------+-------+-------+-------+-------+-------+-------+
| Dist |   0   |  inf  |  inf  |  inf  |  inf  |  inf  |
| Vis. | False | False | False | False | False | False |
| Prev |  None |  None |  None |  None |  None |  None |
+------+-------+-------+-------+-------+-------+-------+

Vertex A is now visited

Neighbour B has estimate decreased from inf to 4 taking a shortcut via A.
Neighbour C has estimate decreased from inf to 4 taking a shortcut via A.

Priority Queue: [(4, 'B'), (4, 'C')]

+------+------+-------+-------+-------+-------+-------+
|      |  A   |   B   |   C   |   D   |   E   |   F   |
+------+------+-------+-------+-------+-------+-------+
| Dist |  0   |   4   |   4   |  inf  |  inf  |  inf  |
| Vis. | True | False | False | False | False | False |
| Prev | None |   A   |   A   |  None |  None 